# Bilingual Internal IT Service Desk — LLM Application Engineering Capstone

**Track:** C — Internal IT Service Desk  
**Step 2:** Architecture, model boundary, configurable backends, and reliability evidence.

> Results are claimed only after the relevant cells execute successfully.

## Architecture target

- Router-first: FAQ single-call; service workflow with tools; terminal human escalation.
- Every model call crosses one `LLMClient` boundary.
- Provider-specific imports exist in exactly one adapter section.
- Two backends are switchable by configuration, not application-code edits.
- Rate-limit and outage fallback are exercised with captured transcripts.

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional
import os
import time

## Common model boundary

In [ ]:
class BackendKind(str, Enum):
    OPEN_WEIGHT = "open_weight"
    COMMERCIAL = "commercial"
    FAKE = "fake"

@dataclass
class LLMRequest:
    messages: List[Dict[str, str]]
    max_tokens: int = 256
    temperature: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class LLMUsage:
    input_tokens: int = 0
    output_tokens: int = 0
    cached_input_tokens: int = 0

@dataclass
class LLMResponse:
    text: str
    backend: str
    model: str
    usage: LLMUsage
    latency_ms: float
    raw: Any = None

class LLMError(RuntimeError): pass
class LLMRateLimitError(LLMError): pass
class LLMBackendUnavailable(LLMError): pass

class LLMClient(ABC):
    backend_kind: BackendKind
    model_name: str

    @abstractmethod
    def generate(self, request: LLMRequest) -> LLMResponse:
        raise NotImplementedError

## Provider adapters — the only provider-import section

`LocalOpenWeightClient` is the no-key default. `CommercialClient` is enabled by configuration when a commercial credential and model name are supplied.

In [ ]:
# === ADAPTER SECTION: PROVIDER-SPECIFIC IMPORTS ARE ALLOWED ONLY HERE ===

class LocalOpenWeightClient(LLMClient):
    backend_kind = BackendKind.OPEN_WEIGHT

    def __init__(self, model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"):
        self.model_name = model_name
        self._pipe = None

    def _ensure_loaded(self):
        from transformers import pipeline
        if self._pipe is None:
            self._pipe = pipeline("text-generation", model=self.model_name, device_map="auto")

    def generate(self, request: LLMRequest) -> LLMResponse:
        self._ensure_loaded()
        start = time.perf_counter()
        prompt = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in request.messages) + "\nASSISTANT:"
        kwargs = dict(max_new_tokens=request.max_tokens, return_full_text=False)
        if request.temperature > 0:
            kwargs.update(do_sample=True, temperature=request.temperature)
        else:
            kwargs.update(do_sample=False)
        out = self._pipe(prompt, **kwargs)
        latency_ms = (time.perf_counter() - start) * 1000
        return LLMResponse(
            text=out[0]["generated_text"].strip(), backend=self.backend_kind.value,
            model=self.model_name, usage=LLMUsage(), latency_ms=latency_ms, raw=out
        )

class CommercialClient(LLMClient):
    backend_kind = BackendKind.COMMERCIAL

    def __init__(self, model_name: str, api_key: Optional[str] = None, base_url: Optional[str] = None):
        from openai import OpenAI
        self.model_name = model_name
        self._client = OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"), base_url=base_url)

    def generate(self, request: LLMRequest) -> LLMResponse:
        start = time.perf_counter()
        result = self._client.responses.create(
            model=self.model_name,
            input=request.messages,
            max_output_tokens=request.max_tokens,
        )
        latency_ms = (time.perf_counter() - start) * 1000
        usage = getattr(result, "usage", None)
        details = getattr(usage, "input_tokens_details", None) if usage else None
        return LLMResponse(
            text=result.output_text,
            backend=self.backend_kind.value,
            model=self.model_name,
            usage=LLMUsage(
                input_tokens=getattr(usage, "input_tokens", 0) if usage else 0,
                output_tokens=getattr(usage, "output_tokens", 0) if usage else 0,
                cached_input_tokens=getattr(details, "cached_tokens", 0) if details else 0,
            ),
            latency_ms=latency_ms,
            raw=result,
        )

# === END ADAPTER SECTION ===

## Config-only backend switching

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    backend: BackendKind = BackendKind.OPEN_WEIGHT
    open_weight_model: str = "Qwen/Qwen2.5-0.5B-Instruct"
    commercial_model: Optional[str] = None
    commercial_base_url: Optional[str] = None

def build_llm_client(config: ModelConfig) -> LLMClient:
    if config.backend == BackendKind.OPEN_WEIGHT:
        return LocalOpenWeightClient(config.open_weight_model)
    if config.backend == BackendKind.COMMERCIAL:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("Commercial backend requested but OPENAI_API_KEY is not configured.")
        if not config.commercial_model:
            raise RuntimeError("Set COMMERCIAL_MODEL before selecting the commercial backend.")
        return CommercialClient(config.commercial_model, base_url=config.commercial_base_url)
    raise ValueError(f"Unsupported backend: {config.backend}")

ACTIVE_CONFIG = ModelConfig(
    backend=BackendKind(os.getenv("LLM_BACKEND", "open_weight")),
    commercial_model=os.getenv("COMMERCIAL_MODEL") or None,
    commercial_base_url=os.getenv("COMMERCIAL_BASE_URL") or None,
)
print("Configured backend:", ACTIVE_CONFIG.backend.value)

## Deterministic fault injection and fallback

In [ ]:
class FakeClient(LLMClient):
    backend_kind = BackendKind.FAKE

    def __init__(self, scripted_events: List[Any], name: str):
        self.scripted_events = list(scripted_events)
        self.model_name = name

    def generate(self, request: LLMRequest) -> LLMResponse:
        if not self.scripted_events:
            raise LLMBackendUnavailable("No scripted response remaining.")
        event = self.scripted_events.pop(0)
        if isinstance(event, Exception):
            raise event
        return LLMResponse(str(event), self.backend_kind.value, self.model_name, LLMUsage(10,5,0), 1.0)

def generate_with_fallback(request: LLMRequest, primary: LLMClient, fallback: LLMClient) -> LLMResponse:
    try:
        print(f"[attempt] primary={primary.model_name}")
        return primary.generate(request)
    except (LLMRateLimitError, LLMBackendUnavailable) as exc:
        print(f"[fallback-triggered] {type(exc).__name__}: {exc}")
        print(f"[attempt] fallback={fallback.model_name}")
        return fallback.generate(request)

## Reliability evidence — both faults must be visibly exercised

In [ ]:
probe = LLMRequest(messages=[{"role":"user","content":"How do I reset my IT password?"}], max_tokens=64)

r1 = generate_with_fallback(
    probe,
    FakeClient([LLMRateLimitError("scripted 429 / rate limit")], "primary-rate-limit-test"),
    FakeClient(["Fallback handled the rate-limit safely."], "fallback-rate-limit-test"),
)
assert "Fallback handled" in r1.text
print("[PASS] rate-limit fallback executed:", r1.text)

print()

r2 = generate_with_fallback(
    probe,
    FakeClient([LLMBackendUnavailable("scripted provider outage")], "primary-outage-test"),
    FakeClient(["Fallback handled the outage safely."], "fallback-outage-test"),
)
assert "Fallback handled" in r2.text
print("[PASS] outage fallback executed:", r2.text)

## Architecture assertion — no provider imports outside the adapter cell

In [ ]:
PROVIDER_IMPORT_PATTERNS = (
    "from " + "openai import", "import " + "openai",
    "from " + "transformers import", "import " + "transformers",
)

def assert_provider_import_boundary(notebook_json: dict) -> None:
    violations = []
    adapter_cells = 0
    for idx, cell in enumerate(notebook_json.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        adapter_marker = "ADAPTER SECTION:" + " PROVIDER-SPECIFIC IMPORTS"
        is_adapter = adapter_marker in source
        if is_adapter:
            adapter_cells += 1
        if any(p in source for p in PROVIDER_IMPORT_PATTERNS) and not is_adapter:
            violations.append(idx)
    assert adapter_cells == 1, f"Expected exactly 1 adapter cell, found {adapter_cells}"
    assert not violations, f"Provider SDK import found outside adapter section in cells: {violations}"
    print("[PASS] exactly one provider-adapter section; no provider imports outside it.")

# Run this against the checked-in notebook file after the repo is cloned/opened.
import json as _json
from pathlib import Path as _Path
_candidate = _Path("notebooks/IT_Service_Desk_Capstone.ipynb")
if _candidate.exists():
    assert_provider_import_boundary(_json.loads(_candidate.read_text(encoding="utf-8")))
else:
    print("[INFO] Boundary test function defined; run it against the checked-in notebook path in Colab.")

## Step-2 status

| Requirement | Status |
|---|---|
| Router-first design | ✅ Designed |
| Common `LLMClient` boundary | ✅ Implemented |
| Config-only backend switch | ✅ Implemented |
| Open-weight adapter | ✅ Implemented |
| Commercial adapter | ✅ Implemented |
| Provider-import assertion | ✅ Implemented |
| Rate-limit fallback drill | ✅ Implemented |
| Outage fallback drill | ✅ Implemented |
| Open-weight backend live run | ⏳ Run in Colab |
| Commercial backend live run | ⏳ Run later with secret |
| Same golden-set comparison | ⏳ Evaluation step |

We do not claim the live-backend points until actual executions are captured.

# Step 3 — Structured outputs, validation, retry, and repair

This section implements the strict domain object required by the service workflow.

**Evidence goals**
- valid English request parses;
- valid Arabic request parses;
- malformed model output is rejected;
- retry is attempted;
- repair is attempted if retry remains invalid;
- final object is strictly validated by Pydantic.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

class ITServiceRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    request_type: Literal[
        "access_request",
        "asset_booking",
        "incident",
        "status_check",
    ]
    target: str = Field(min_length=1, max_length=120)
    justification: str | None = Field(default=None, max_length=500)
    urgency: Literal["low", "medium", "high", "critical"]
    security_sensitive: bool
    language: Literal["ar", "en"]

print("[PASS] ITServiceRequest schema loaded with strict validation.")

## Structured parsing helper

The application never trusts raw model JSON. It must pass schema validation before the workflow can continue.

In [ ]:
import json

def parse_it_service_request(raw_text: str) -> ITServiceRequest:
    payload = json.loads(raw_text)
    return ITServiceRequest.model_validate(payload)

## Bilingual happy-path evidence

In [ ]:
english_request = json.dumps({
    "request_type": "access_request",
    "target": "Finance Analytics Portal",
    "justification": "Required for monthly reporting duties",
    "urgency": "medium",
    "security_sensitive": False,
    "language": "en",
})

arabic_request = json.dumps({
    "request_type": "incident",
    "target": "VPN",
    "justification": "لا أستطيع الاتصال بالشبكة من خارج المكتب",
    "urgency": "high",
    "security_sensitive": False,
    "language": "ar",
}, ensure_ascii=False)

parsed_en = parse_it_service_request(english_request)
parsed_ar = parse_it_service_request(arabic_request)

assert parsed_en.language == "en"
assert parsed_ar.language == "ar"
assert parsed_en.request_type == "access_request"
assert parsed_ar.request_type == "incident"

print("[PASS] English structured request validated:")
print(parsed_en.model_dump())
print()
print("[PASS] Arabic structured request validated:")
print(parsed_ar.model_dump())

## Validate → retry → repair loop

The deterministic test below deliberately emits two invalid attempts followed by a valid repaired object.

In [ ]:
from dataclasses import dataclass

@dataclass
class StructuredAttempt:
    stage: str
    raw_text: str
    valid: bool
    error: str | None = None

def validate_retry_repair(initial_raw, retry_fn, repair_fn):
    attempts = []

    def attempt(stage, raw):
        try:
            parsed = parse_it_service_request(raw)
            attempts.append(StructuredAttempt(stage, raw, True, None))
            return parsed
        except Exception as exc:
            attempts.append(StructuredAttempt(stage, raw, False, f"{type(exc).__name__}: {exc}"))
            return None

    parsed = attempt("initial", initial_raw)
    if parsed is not None:
        return parsed, attempts

    retry_raw = retry_fn()
    parsed = attempt("retry", retry_raw)
    if parsed is not None:
        return parsed, attempts

    repair_raw = repair_fn(initial_raw, retry_raw)
    parsed = attempt("repair", repair_raw)
    if parsed is None:
        raise RuntimeError("Structured output remained invalid after repair.")
    return parsed, attempts

## Deliberate failure-and-repair demonstration

In [ ]:
malformed_initial = '''
{
  "request_type": "access_request",
  "target": "HR System",
  "justification": "Need access for work",
  "urgency": "urgent",
  "security_sensitive": "no",
  "language": "en"
}
'''

def scripted_retry():
    return json.dumps({
        "request_type": "admin_override",
        "target": "HR System",
        "justification": "Need access for work",
        "urgency": "high",
        "security_sensitive": False,
        "language": "en",
        "role": "administrator",
    })

def scripted_repair(initial_raw, retry_raw):
    return json.dumps({
        "request_type": "access_request",
        "target": "HR System",
        "justification": "Need access for work",
        "urgency": "high",
        "security_sensitive": False,
        "language": "en",
    })

final_request, attempt_log = validate_retry_repair(
    malformed_initial,
    scripted_retry,
    scripted_repair,
)

for item in attempt_log:
    print(
        f"[{item.stage.upper()}] valid={item.valid}"
        + (f" | {item.error.splitlines()[0]}" if item.error else "")
    )

assert [a.valid for a in attempt_log] == [False, False, True]
assert final_request.request_type == "access_request"
assert final_request.urgency == "high"
assert final_request.security_sensitive is False

print()
print("[PASS] validate -> retry -> repair executed successfully.")
print("[PASS] Final strictly validated object:")
print(final_request.model_dump())

## Step-3 acceptance checklist

| Requirement | Evidence |
|---|---|
| Strict Pydantic schema | ✅ implemented |
| Extra fields forbidden | ✅ implemented |
| English parse | ✅ executable test |
| Arabic parse | ✅ executable test |
| Invalid output rejected | ✅ executable test |
| Retry occurs | ✅ executable test |
| Repair occurs | ✅ executable test |
| Final object strictly validated | ✅ executable assertion |

Identity and authorization are intentionally absent from this schema. Those come from authenticated session state in Step 4.